# When can a force pulse be replaced by its impulse?

Keep the total impulse fixed and increase contact duration. Predict whether the free vibration after release stays constant. Compare the Fourier integral with a direct quadrature of the forcing history.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

mass,frequency=2.,8.    # kg, Hz
impulse= .6            # N s
duration=.050          # s
assert min(mass,frequency,impulse,duration)>0
omega=2*np.pi*frequency

## Project the pulse on the free oscillator
Use two shifted sinc functions to evaluate the normalised half-sine pulse transform without a removable singularity. NumPy sinc uses sin(pi x)/(pi x).

In [ ]:
def ratio(r):
    return np.pi/4*np.abs(np.sinc((r-np.pi)/(2*np.pi))+np.sinc((r+np.pi)/(2*np.pi)))

u=np.linspace(0,1,801)
weights=np.pi/2*np.sin(np.pi*u)
r=omega*duration
q=weights*np.exp(-1j*r*u)
integrated=np.sum(np.diff(u)*(q[:-1]+q[1:])/2)
assert abs(abs(integrated)-ratio(r))<4e-6
assert np.isclose(ratio(0),1) and np.isclose(ratio(np.pi),np.pi/4)
amplitude=impulse/(mass*omega)*ratio(r)
print(f"Pulse / natural period {frequency*duration:.3f}; remaining amplitude {1000*amplitude:.3f} mm")

## Read the shape through its residual motion
At the same impulse, a long pulse is weaker and excites less residual motion over many duration ranges. The first exact cancellation occurs at one and a half periods.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(9,3.5),layout="constrained")
for factor,style in [(1,"-"),(2,"--")]:
    T=duration*factor
    ax[0].plot(u*T*1000,impulse/T*weights,style,label=f"{T*1000:.0f} ms")
ax[0].set(xlabel="Time after contact (ms)",ylabel="Force (N)",title="Equal force impulse")
cycles=np.linspace(0,3,550)
ax[1].plot(cycles,ratio(2*np.pi*cycles),label="Remaining / impulsive amplitude")
ax[1].plot(frequency*duration,ratio(r),"o",label="Chosen duration")
ax[1].set(xlabel="Pulse duration / natural period",ylabel="Residual amplitude ratio",title="Free vibration after release")
for p in ax: p.grid(alpha=.22);p.legend(fontsize=8)
plt.show()

Change the duration to half a period and then one and a half periods. Why does one case leave a sizeable oscillation while the other cancels it? Explain why cancellation after release does not imply zero displacement during contact.